# Mixed Precision Training

by default PyTorch uses float32 (32-bit). Using float16 or bfloat16 (16-bit) gives:
- **2x less GPU memory** — fit larger models / larger batches
- **2-3x faster** — GPU tensor cores optimized for 16-bit ops

**Mixed precision** = keep weights in float32 for stability, compute forward/backward in float16 for speed.

| dtype | Bits | Range | Use case |
|---|---|---|---|
| float32 | 32 | ±3.4×10³⁸ | Default, stable |
| float16 | 16 | ±65,504 | Older GPUs (V100, T4) — can overflow |
| bfloat16 | 16 | ±3.4×10³⁸ | Modern GPUs (A100, H100) + TPUs — same range as float32 |

In [ ]:
import torch
import torch.nn as nn

# Check bfloat16 support (requires Ampere+ GPU or TPU)
print('bfloat16 support:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else 'N/A (no GPU)')

# --- torch.autocast — automatic mixed precision ---
# Wraps forward pass: PyTorch automatically chooses float16 or float32 per operation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model   = nn.Linear(512, 512).to(device)
x       = torch.randn(32, 512).to(device)

# Without mixed precision
out = model(x)
print('Default dtype:', out.dtype)   # float32

# With mixed precision — autocast handles the conversion
with torch.autocast(device_type=device, dtype=dtype):
    out_amp = model(x)
print('AMP dtype:     ', out_amp.dtype)   # float16 or bfloat16

In [ ]:
# --- GradScaler — needed with float16 (NOT needed with bfloat16) ---
# float16 has a small range — gradients can underflow to 0
# GradScaler multiplies the loss by a large scale factor before backward
# then divides gradients back before optimizer step

from torch.cuda.amp import GradScaler

scaler    = GradScaler(enabled=(dtype == torch.float16))  # only for float16
model     = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(3):
    x      = torch.randn(32, 512).to(device)
    labels = torch.randint(0, 10, (32,)).to(device)

    optimizer.zero_grad()

    with torch.autocast(device_type=device, dtype=dtype):
        logits = model(x)           # forward in float16
        loss   = loss_fn(logits, labels)

    scaler.scale(loss).backward()   # backward with scaled gradients
    scaler.step(optimizer)          # unscale + optimizer step
    scaler.update()                 # adjust scale for next step

    print(f'Step {step+1} | loss: {loss.item():.4f}')

## Practical rule

```
GPU type         → dtype to use
A100, H100, TPU  → bfloat16  (no GradScaler needed)
T4, V100, 3090   → float16   (need GradScaler)
No GPU / CPU     → float32   (default)
```

In HuggingFace Trainer, this is one line:
```python
TrainingArguments(bf16=True)   # or fp16=True for older GPUs
```